### Dataset & EDA

In [1]:
!pip install google.colab

ERROR: Could not find a version that satisfies the requirement google.colab (from versions: none)
ERROR: No matching distribution found for google.colab


In [2]:
import joblib
from google.colab import files
files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download clmentbisaillon/fake-and-real-news-dataset

In [ ]:
!ls /content

In [ ]:
!unzip fake-and-real-news-dataset.zip

In [ ]:
import pandas as pd

In [ ]:
true = pd.read_csv('/content/True.csv')

In [ ]:
true.head(3)

In [ ]:
fake = pd.read_csv('/content/Fake.csv')

In [ ]:
fake.head(3)

In [ ]:
true.shape

In [ ]:
fake.shape

In [ ]:
true['label'] = 1

In [ ]:
true.head(3)

In [ ]:
fake['label'] = 0

In [ ]:
fake.head(3)

In [ ]:
frames = [true.loc[0:5000][:], fake.loc[0:5000][:]]
df = pd.concat(frames)

In [ ]:
df.shape

In [ ]:
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
X = df.drop('label', axis=1)

In [ ]:
y = df['label']

In [ ]:
df = df.dropna()

In [ ]:
messages = df.copy()

In [ ]:
messages.reset_index(inplace=True)

In [ ]:
messages.head()

In [ ]:
messages['title'][2]

In [ ]:
messages['text'][2]

### Preprocessing

In [ ]:
# =========================
# TEXT COLUMN DETECTION
# =========================
text_column = None
for col in X.columns:
    if X[col].dtype == "object":
        text_column = col
        break

# =========================
# FINAL INPUT FIX
# =========================
if tfidf is not None and text_column is not None:
    st.info("🧠 Using TF-IDF vectorizer")

    # 🔥 SAME preprocessing as notebook
    import re
    from nltk.corpus import stopwords
    from nltk.stem.porter import PorterStemmer

    ps = PorterStemmer()

    corpus = []
    for i in range(len(X)):
        review = re.sub('[^a-zA-Z]', ' ', str(X[text_column].iloc[i]))
        review = review.lower()
        review = review.split()
        review = [ps.stem(word) for word in review if word not in stopwords.words('english')]
        review = ' '.join(review)
        corpus.append(review)

    X_input = tfidf.transform(corpus).toarray()

elif isinstance(model, Pipeline):
    X_input = X[text_column]

else:
    # fallback numeric (your original)
    X_clean = X.copy()

    for col in X_clean.columns:
        try:
            X_clean[col] = pd.to_numeric(X_clean[col])
        except:
            X_clean[col] = pd.factorize(X_clean[col])[0]

    X_clean = X_clean.fillna(0)
    X_input = X_clean.values.astype(np.float32)

### Passive Aggressive Classifier - Baseline

In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier
classifier = PassiveAggressiveClassifier(max_iter=1000)

In [ ]:
from sklearn import metrics
import numpy as np
import itertools

classifier.fit(X_train, y_train)

pred = classifier.predict(X_test)

score = metrics.accuracy_score(y_test, pred)
print("accuracy:   %0.3f" % score)

In [ ]:
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    See full source and example: 
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html
    
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
cm = metrics.confusion_matrix(y_test, pred)
plot_confusion_matrix(cm, classes=['FAKE', 'REAL'])

In [ ]:
classifier.predict(X_test)

### Preprocess and transform datapoint text (true['text'][16888]) and try to predict it based on the model

In [ ]:
review = re.sub('[^a-zA-Z]', ' ', true['text'][16888])
review = review.lower()
review = review.split()
    
review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
review = ' '.join(review)
review

In [ ]:
val = tfidf_v.transform([review]).toarray()

In [ ]:
pd.DataFrame(val, columns=tfidf_v.get_feature_names())

In [ ]:
classifier.predict(val)

### Saving model and TFIDF Vectorizer

In [ ]:
from sklearn.externals import joblib

In [ ]:
joblib.dump(classifier, 'model.pkl')

In [ ]:
joblib.dump(tfidf_v, 'tfidfvect.pkl')

### Load model and vectorizer and predict on previous preprocessed datapoint

In [ ]:
joblib_model = joblib.load('model.pkl')

In [ ]:
joblib_tfidfvect = joblib.load('tfidfvect.pkl')

In [ ]:
val_pkl = joblib_tfidfvect.transform([review]).toarray()

In [ ]:
joblib_model.predict(val_pkl)

### Save some datapoints for text random generation

In [ ]:
frames_2 = [true.loc[0:10][:], fake.loc[0:10][:]]
df_2 = pd.concat(frames_2)

In [ ]:
df_2.to_csv('random_dataset.csv', index=False)